# Synthetic Trade Generator

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

class SyntheticTradeGenerator:
    """
    Generates simulated Front Office (FO) and Back Office (BO) trade feeds.
    """
    
    def __init__(self, num_trades=100, random_state=42):
        self.num_trades = num_trades
        self.random_state = random_state
        
        # Lock in the random seed for reproducible datasets
        np.random.seed(self.random_state)
        
        self.product_types = ['Equity Index Option', 'Total Return Swap', 'Stock Loan']
        self.counterparties = ['VectaLink Global Ltd.', 'Citadel', 'BlackRock', 'Millennium']

    def _generate_base_trades(self):
        """Generates the clean Front Office dataset."""
        # Vectorized generation for speed and clean syntax
        days_ago = np.random.randint(0, 10, self.num_trades)
        trade_dates = [datetime.now() - timedelta(days=int(d)) for d in days_ago]
        
        data = {
            'Trade_ID': [f"TRD-{1000 + i}" for i in range(self.num_trades)],
            'Product_Type': np.random.choice(self.product_types, self.num_trades),
            'Counterparty': np.random.choice(self.counterparties, self.num_trades),
            'Notional_GBP': np.round(np.random.uniform(500000, 5000000, self.num_trades), 2),
            'Strike_Price': np.round(np.random.uniform(100, 500, self.num_trades), 2),
            'MTM_GBP': np.round(np.random.uniform(-100000, 100000, self.num_trades), 2),
            'Day_Count_Basis': np.where(np.random.rand(self.num_trades) > 0.5, 'Actual/365', '30/360'),
            'Trade_Date': [d.strftime('%Y-%m-%d') for d in trade_dates],
            'Status': ['Confirmed'] * self.num_trades
        }
        
        return pd.DataFrame(data)

    def generate_feeds(self):
        # Create pure FO feed
        fo_df = self._generate_base_trades()
        
        # Clone FO feed to create BO feed, then inject errors
        bo_df = fo_df.copy()
        
        # Select 15 unique random indices for error injection
        break_indices = np.random.choice(self.num_trades, size=15, replace=False)
        
        # Break 1: Accrual Mismatch
        for idx in break_indices[0:5]:
            current_basis = fo_df.at[idx, 'Day_Count_Basis']
            bo_df.at[idx, 'Day_Count_Basis'] = '30/360' if current_basis == 'Actual/365' else 'Actual/365'
            
        # Break 2: MTM Discrepancy 
        for idx in break_indices[5:10]:
            error_margin = np.random.uniform(50000, 150000)
            bo_df.at[idx, 'MTM_GBP'] = fo_df.at[idx, 'MTM_GBP'] + error_margin
            
        # Break 3: Unconfirmed Status 
        for idx in break_indices[10:15]:
            bo_df.at[idx, 'Status'] = 'Unconfirmed'
            
        # Break 4: Missing Trades in BO (Drop 2 random rows entirely)
        drop_indices = np.random.choice(self.num_trades, size=2, replace=False)
        bo_df = bo_df.drop(drop_indices)
        
        return fo_df, bo_df

In [6]:
# Instantiate the class and generate the data
generator = SyntheticTradeGenerator(num_trades=100)
fo_feed, bo_feed = generator.generate_feeds()

# Check the shapes (BO should have 98 rows due to the 2 dropped captures)
print(f"FO Feed Shape: {fo_feed.shape}")
print(f"BO Feed Shape: {bo_feed.shape}")

# Export for Module 2
fo_feed.to_csv('fo_feed.csv', index=False)
bo_feed.to_csv('bo_feed.csv', index=False)

FO Feed Shape: (100, 9)
BO Feed Shape: (98, 9)


In [11]:
class TradeReconciler:
    """
    Reconciles Front Office (FO) and Back Office (BO) trade feeds using vectorized logic
    and quantifies Exposure at Risk (EaR) for exception prioritization.
    """
    
    def __init__(self, fo_feed, bo_feed):
        self.fo_df = fo_feed
        self.bo_df = bo_feed

    def _join_feeds(self):
        """Joins FO and BO feeds on Trade_ID for reconciliation."""
        return pd.merge(self.fo_df, self.bo_df, on='Trade_ID', how='outer', suffixes=('_FO', '_BO'))

    def detect_breaks(self):
        """Identifies exceptions, calculates Exposure at Risk (GBP), and assigns severity."""
        df = self._join_feeds()
        df['Break_Type'] = 'Matched'
        
        # 1. Vectorized Break Detection
        df.loc[df['MTM_GBP_BO'].isna(), 'Break_Type'] = 'Missing in BO'
        df.loc[df['MTM_GBP_FO'].isna(), 'Break_Type'] = 'Missing in FO'
        
        df['MTM_Diff'] = (df['MTM_GBP_FO'] - df['MTM_GBP_BO']).abs()
        mtm_mask = (df['Break_Type'] == 'Matched') & (df['MTM_Diff'] > 1000)
        df.loc[mtm_mask, 'Break_Type'] = 'MTM Discrepancy > £1k'
        
        basis_mask = (df['Break_Type'] == 'Matched') & (df['Day_Count_Basis_FO'] != df['Day_Count_Basis_BO'])
        df.loc[basis_mask, 'Break_Type'] = 'Accrual Basis Mismatch'
        
        status_mask = (df['Break_Type'] == 'Matched') & (df['Status_BO'] == 'Unconfirmed')
        df.loc[status_mask, 'Break_Type'] = 'Unconfirmed in BO'
        
        # Filter down to only the breaks
        breaks_df = df[df['Break_Type'] != 'Matched'].copy()
        
        # 2. Vectorized Exposure at Risk (EaR) Calculation via np.select
        conditions = [
            breaks_df['Break_Type'] == 'Missing in BO',
            breaks_df['Break_Type'] == 'MTM Discrepancy > £1k',
            breaks_df['Break_Type'] == 'Unconfirmed in BO',
            breaks_df['Break_Type'] == 'Accrual Basis Mismatch'
        ]
        
        # Financial impact definitions per break type
        choices = [
            breaks_df['MTM_GBP_FO'].abs(),                  # Full unhedged/unbooked MTM
            breaks_df['MTM_Diff'],                          # Unexplained valuation gap
            breaks_df['MTM_GBP_FO'].abs(),                  # Counterparty legal/confirmation risk
            (breaks_df['Notional_GBP_FO'] * 0.05) * (5/365) # 5-day basis spread proxy at 5% rate
        ]
        
        breaks_df['Exposure_at_Risk_GBP'] = np.round(np.select(conditions, choices, default=0.0), 2)
        
        # 3. Break Aging (Days Open) & Escalation Severity
        today = pd.to_datetime(datetime.now().strftime('%Y-%m-%d'))
        breaks_df['Age_Days'] = (today - pd.to_datetime(breaks_df['Trade_Date_FO'])).dt.days
        
        # Escalate if Exposure > £50k, Missing in BO, or Aged > 5 days
        crit_mask = (
            (breaks_df['Exposure_at_Risk_GBP'] > 50000) | 
            (breaks_df['Break_Type'] == 'Missing in BO') | 
            (breaks_df['Age_Days'] > 5)
        )
        breaks_df['Severity'] = np.where(crit_mask, 'CRITICAL', 'HIGH')
        
        # Sort by Severity and highest Exposure at Risk
        return breaks_df.sort_values(by=['Exposure_at_Risk_GBP'], ascending=False)

    def generate_executive_summary(self, breaks_df):
        """Aggregates break metrics by Break_Type for Control Desk reporting."""
        summary = breaks_df.groupby('Break_Type').agg(
            Break_Count=('Trade_ID', 'count'),
            Total_Exposure_GBP=('Exposure_at_Risk_GBP', 'sum'),
            Max_Age_Days=('Age_Days', 'max')
        ).reset_index()
        return summary.sort_values(by='Total_Exposure_GBP', ascending=False)

In [12]:
reconciler = TradeReconciler(fo_feed, bo_feed)
reconciliation_breaks = reconciler.detect_breaks()

# 1. View the Executive Control Summary
exec_summary = reconciler.generate_executive_summary(reconciliation_breaks)
print("--- EXECUTIVE RECONCILIATION SUMMARY ---")
display(exec_summary)

# 2. View the Prioritized Exception Queue
print("\n--- PRIORITIZED EXCEPTION QUEUE ---")
cols = ['Trade_ID', 'Counterparty_FO', 'Product_Type_FO', 'Break_Type', 'Exposure_at_Risk_GBP', 'Age_Days', 'Severity']
display(reconciliation_breaks[cols])

--- EXECUTIVE RECONCILIATION SUMMARY ---


,Break_Type,Break_Count,Total_Exposure_GBP,Max_Age_Days
1,MTM Discrepancy > £1k,5,451972.56,9
3,Unconfirmed in BO,5,203759.54,7
2,Missing in BO,2,159794.61,9
0,Accrual Basis Mismatch,5,11473.62,8



--- PRIORITIZED EXCEPTION QUEUE ---


,Trade_ID,Counterparty_FO,Product_Type_FO,Break_Type,Exposure_at_Risk_GBP,Age_Days,Severity
53,TRD-1053,Millennium,Equity Index Option,MTM Discrepancy > £1k,143369.18,1,CRITICAL
5,TRD-1005,Millennium,Equity Index Option,MTM Discrepancy > £1k,98780.98,9,CRITICAL
78,TRD-1078,Citadel,Equity Index Option,Unconfirmed in BO,98592.96,0,CRITICAL
87,TRD-1087,Citadel,Equity Index Option,MTM Discrepancy > £1k,86535.68,0,CRITICAL
52,TRD-1052,BlackRock,Equity Index Option,Missing in BO,83299.86,3,CRITICAL
63,TRD-1063,Millennium,Equity Index Option,Missing in BO,76494.75,9,CRITICAL
30,TRD-1030,Citadel,Stock Loan,MTM Discrepancy > £1k,72533.28,8,CRITICAL
32,TRD-1032,BlackRock,Stock Loan,MTM Discrepancy > £1k,50753.44,4,CRITICAL
38,TRD-1038,Millennium,Stock Loan,Unconfirmed in BO,40193.83,1,HIGH
84,TRD-1084,Citadel,Equity Index Option,Unconfirmed in BO,39149.74,7,CRITICAL


In [13]:
class ResolutionRouter:
    """
    Routes prioritized reconciliation breaks to target operational desks,
    assigns SLAs, and generates structured investigation payloads.
    """
    
    def __init__(self, breaks_df):
        self.breaks_df = breaks_df.copy()
        
        # Operational routing matrix
        self.routing_map = {
            'Missing in BO': 'Middle Office Trade Capture',
            'Missing in FO': 'Front Office Desk Support',
            'MTM Discrepancy > £1k': 'Valuation Control (IPV) & Product Control',
            'Unconfirmed in BO': 'Client Confirmations & Legal Settlements',
            'Accrual Basis Mismatch': 'Static Data & Middle Office Operations'
        }
        
        # Root-cause remediation playbooks
        self.action_map = {
            'Missing in BO': 'Re-trigger MQ trade message from FO blotter; verify booking schema.',
            'Missing in FO': 'Halt settlement; verify potential duplicate orrogue BO booking.',
            'MTM Discrepancy > £1k': 'Compare FO vol surface / curve marks against BO end-of-day snap.',
            'Unconfirmed in BO': 'Chase counterparty affirmation via MarkitWire / Swift; escalate to Sales.',
            'Accrual Basis Mismatch': 'Align day-count convention against master ISDA/CSA confirmation.'
        }

    def route_exceptions(self):
        """Applies vectorized routing, SLA assignment, and diagnostic tickets."""
        df = self.breaks_df
        
        # 1. Vectorized Desk & Playbook Mapping
        df['Assigned_Desk'] = df['Break_Type'].map(self.routing_map)
        df['Recommended_Action'] = df['Break_Type'].map(self.action_map)
        
        # 2. Vectorized SLA Assignment (Hours) based on Severity and Break Type
        sla_conditions = [
            df['Break_Type'] == 'Missing in BO',
            df['Severity'] == 'CRITICAL',
            df['Severity'] == 'HIGH'
        ]
        sla_Dynamic_Hours = [2, 4, 24] # T+0 2-hr fix for dropped captures, 4-hr for Critical, 24-hr for High
        df['SLA_Hours'] = np.select(sla_conditions, sla_Dynamic_Hours, default=24)
        
        # 3. Generate Structured Audit Ticket String
        df['Audit_Ticket'] = (
            "[" + df['Severity'] + " | SLA: " + df['SLA_Hours'].astype(str) + "h] " +
            df['Trade_ID'] + " (" + df['Counterparty_FO'].fillna('Unknown') + " - " + 
            df['Product_Type_FO'].fillna('Unknown') + "): " + df['Break_Type'] + 
            " | EaR: £" + df['Exposure_at_Risk_GBP'].apply(lambda x: f"{x:,.2f}") + 
            " | Age: " + df['Age_Days'].astype(str) + "d -> Route to: " + df['Assigned_Desk']
        )
        
        return df

    def export_control_pack(self, routed_df, filename='ed_control_exceptions_pack.csv'):
        """Exports the final exception queue for downstream dashboarding or agent ingestion."""
        export_cols = [
            'Trade_ID', 'Counterparty_FO', 'Product_Type_FO', 'Notional_GBP_FO',
            'Break_Type', 'Exposure_at_Risk_GBP', 'Age_Days', 'Severity',
            'Assigned_Desk', 'SLA_Hours', 'Recommended_Action', 'Audit_Ticket'
        ]
        routed_df[export_cols].to_csv(filename, index=False)
        return export_cols

In [14]:
# Run the Resolution Router
router = ResolutionRouter(reconciliation_breaks)
routed_exceptions = router.route_exceptions()
exported_cols = router.export_control_pack(routed_exceptions)

# 1. Print the Top 5 Critical Automated Tickets
print("--- TOP 5 AUTOMATED ESCALATION TICKETS ---")
for ticket in routed_exceptions['Audit_Ticket'].head(5):
    print(ticket)

# 2. Display Desk Workload Breakdown
print("\n--- WORKLOAD BY ASSIGNED DESK ---")
desk_summary = routed_exceptions.groupby('Assigned_Desk').agg(
    Open_Tickets=('Trade_ID', 'count'),
    Critical_Count=('Severity', lambda x: (x == 'CRITICAL').sum()),
    Desk_Exposure_GBP=('Exposure_at_Risk_GBP', 'sum')
).reset_index().sort_values(by='Desk_Exposure_GBP', ascending=False)
display(desk_summary)

--- TOP 5 AUTOMATED ESCALATION TICKETS ---
[CRITICAL | SLA: 4h] TRD-1053 (Millennium - Equity Index Option): MTM Discrepancy > £1k | EaR: £143,369.18 | Age: 1d -> Route to: Valuation Control (IPV) & Product Control
[CRITICAL | SLA: 4h] TRD-1005 (Millennium - Equity Index Option): MTM Discrepancy > £1k | EaR: £98,780.98 | Age: 9d -> Route to: Valuation Control (IPV) & Product Control
[CRITICAL | SLA: 4h] TRD-1078 (Citadel - Equity Index Option): Unconfirmed in BO | EaR: £98,592.96 | Age: 0d -> Route to: Client Confirmations & Legal Settlements
[CRITICAL | SLA: 4h] TRD-1087 (Citadel - Equity Index Option): MTM Discrepancy > £1k | EaR: £86,535.68 | Age: 0d -> Route to: Valuation Control (IPV) & Product Control
[CRITICAL | SLA: 2h] TRD-1052 (BlackRock - Equity Index Option): Missing in BO | EaR: £83,299.86 | Age: 3d -> Route to: Middle Office Trade Capture

--- WORKLOAD BY ASSIGNED DESK ---


,Assigned_Desk,Open_Tickets,Critical_Count,Desk_Exposure_GBP
3,Valuation Control (IPV) & Product Control,5,5,451972.56
0,Client Confirmations & Legal Settlements,5,4,203759.54
1,Middle Office Trade Capture,2,2,159794.61
2,Static Data & Middle Office Operations,5,1,11473.62
